# 05 · Near-duplicate detection with MinHash and LSH

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/pretraining-data/05_minhash_lsh.ipynb) &nbsp; 📖 Concept deck: [Deduplication](https://nilesh-salpe.github.io/visual-ai/pretraining-data/07-deduplication/) &nbsp; ⏱️ Suggested time: **40 min**

**You will build:** word n-gram shingles and exact Jaccard similarity, MinHash signatures whose agreement estimates Jaccard, and locality-sensitive hashing with b bands of r rows, including its S-shaped detection curve.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


import hashlib, itertools
BASE = ("the quick brown fox jumps over the lazy dog while the farmer watches from the old wooden barn "
        "and the sun sets slowly behind the hills casting long shadows across the quiet green fields").split()
def variant(k, seed):
    """BASE with k random word substitutions."""
    r = np.random.default_rng(seed); w = list(BASE)
    for i in r.choice(len(w), size=k, replace=False): w[i] = f"word{r.integers(1e6)}"
    return " ".join(w)
P = (1 << 61) - 1
def h64(s): return int.from_bytes(hashlib.blake2b(s.encode(), digest_size=8).digest(), "big")

## Task 1: Shingles and Jaccard similarity

`shingles(text, n)` returns the set of word n-grams (as tuples). `jaccard(a, b)` = |A ∩ B| / |A ∪ B|. Return the two functions as a tuple.

In [ ]:
def shingles(text, n=5):
    # TODO
    raise NotImplementedError
def jaccard(A, B):
    # TODO
    raise NotImplementedError
def shingles_jaccard():
    return shingles, jaccard

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_shingles_jaccard(f):
    sh, jac = f()
    a = sh("a b c d e f", 3)
    assert a == {("a", "b", "c"), ("b", "c", "d"), ("c", "d", "e"), ("d", "e", "f")}
    assert jac(a, a) == 1 and jac(a, set()) == 0
    assert 0.3 < jac(sh(variant(2, 0)), sh(" ".join(BASE))) < 0.9

check("shingles_jaccard", _test_shingles_jaccard, shingles_jaccard)

## Task 2: MinHash signatures

Hash each shingle to a 64-bit integer with `h64`, then apply `k` random affine hash functions $h_i(x) = (a_i x + b_i) \bmod P$ and keep the **minimum** over the set for each i. Return the signature array of length k.
The probability that two sets agree on one minimum equals their Jaccard similarity (Broder, 1997), so the fraction of agreeing positions estimates it.

In [ ]:
def minhash(shingle_set, k=128, seed=0):
    r = np.random.default_rng(seed)
    a = r.integers(1, P, size=k, dtype=np.uint64); b = r.integers(0, P, size=k, dtype=np.uint64)
    # TODO: use Python ints for the modular arithmetic (np.uint64 overflows); return np.array of mins
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_minhash(f):
    sh = lambda t, n=5: {tuple(t.split()[i:i + n]) for i in range(len(t.split()) - n + 1)}
    jac = lambda X, Y: len(X & Y) / len(X | Y)
    A, B = sh(" ".join(BASE)), sh(variant(3, 1))
    sa, sb = f(A, 256), f(B, 256)
    assert len(sa) == 256
    est, true = np.mean(sa == sb), jac(A, B)
    assert abs(est - true) < 0.1, f"MinHash estimate {est:.2f} should be close to Jaccard {true:.2f}"
    assert np.all(f(A, 64) == f(A, 64)), "deterministic for a fixed seed"

check("minhash", _test_minhash, minhash)

## Task 3: LSH banding: the S-curve

Split a signature of b·r values into **b bands of r rows**. Two documents become candidates if **all r rows of at least one band** match. With Jaccard s, that happens with probability $P(s) = 1 - (1 - s^r)^b$.
Return P for the given s, r, b. Check: RefinedWeb's 450 bands × 20 rows give P(0.75) ≈ 0.76 and P(0.8) ≈ 0.994.

In [ ]:
def lsh_prob(s, r, b):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_lsh_prob(f):
    assert abs(f(0.75, 20, 450) - 0.76) < 0.01 and abs(f(0.8, 20, 450) - 0.994) < 0.002, "RefinedWeb's numbers"
    assert f(0.3, 8, 14) < 0.01 and f(0.9, 8, 14) > 0.99, "FineWeb's 14 bands × 8 rows: steep around 0.75"

check("lsh_prob", _test_lsh_prob, lsh_prob)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Shingles and Jaccard similarity
def shingles_s(text, n=5):
    w = text.split(); return {tuple(w[i:i + n]) for i in range(len(w) - n + 1)}
def jaccard_s(A, B):
    return len(A & B) / len(A | B) if A | B else 0.0
def shingles_jaccard_solution():
    return shingles_s, jaccard_s

check("shingles_jaccard (reference)", _test_shingles_jaccard, shingles_jaccard_solution)

In [ ]:
# @title Solution for Task 2: MinHash signatures
def minhash_solution(shingle_set, k=128, seed=0):
    r = np.random.default_rng(seed)
    a = [int(x) for x in r.integers(1, P, size=k, dtype=np.uint64)]; b = [int(x) for x in r.integers(0, P, size=k, dtype=np.uint64)]
    xs = [h64(" ".join(s)) for s in shingle_set]
    return np.array([min((ai * x + bi) % P for x in xs) for ai, bi in zip(a, b)], dtype=object)

check("minhash (reference)", _test_minhash, minhash_solution)

In [ ]:
# @title Solution for Task 3: LSH banding: the S-curve
def lsh_prob_solution(s, r, b):
    return 1 - (1 - s ** r) ** b

check("lsh_prob (reference)", _test_lsh_prob, lsh_prob_solution)

---
## Experiments
**Find near-duplicates in a small corpus with banding.** Build 40 documents (variants of one base text with 0–12 substitutions, plus unrelated random documents), MinHash them, bucket by band, and compare candidates with true Jaccard.

In [ ]:
sh, jac = shingles_jaccard_solution()
docs = [variant(k, s) for s, k in enumerate([0, 1, 2, 3, 5, 8, 12] * 3)] + [" ".join(np.random.default_rng(100 + i).choice(["x%d" % j for j in range(500)], 30)) for i in range(19)]
S = [sh(d) for d in docs]; sig = [minhash_solution(s, 112) for s in S]
b, r = 14, 8; buckets = {}
for i, sgn in enumerate(sig):
    for band in range(b):
        buckets.setdefault((band, tuple(sgn[band * r:(band + 1) * r])), []).append(i)
cand = {tuple(sorted(p)) for ids in buckets.values() for p in itertools.combinations(ids, 2)}
sims = [jac(S[i], S[j]) for i, j in cand]
allpairs = [jac(S[i], S[j]) for i, j in itertools.combinations(range(len(S)), 2)]
print(f"{len(cand)} candidate pairs out of {len(allpairs)}; min Jaccard among candidates {min(sims):.2f}")
print(f"true pairs with Jaccard ≥ 0.75: {sum(s >= 0.75 for s in allpairs)}, found: {sum(s >= 0.75 for s in sims)}")

---
## Takeaways
* Near-duplicates are measured by Jaccard similarity of word n-gram sets.
* MinHash compresses each document to k numbers whose agreement rate estimates Jaccard.
* LSH banding finds candidate pairs without comparing all pairs; b and r place the S-curve's threshold (≈ (1/b)^(1/r)).